In [1]:
import os
import subprocess
import re
import pandas as pd
import time
import numpy as np

In [2]:
os.environ["SLURM_TIME_FORMAT"] = "%s"

In [3]:
def get_data_from_sacct(clusters: str,
                        partition: str,
                        start_date: str,
                        end_date: str,
                        fields: str) -> pd.DataFrame:
    """Return a dataframe of the sacct output."""
    cmd = f"sacct -M {clusters} -u ey7932 -X -P -n -S {start_date} -E {end_date} -o {fields}"
    if partition:
        cmd += f" -r {partition}"
    output = subprocess.run(cmd,
                            stdout=subprocess.PIPE,
                            shell=True,
                            timeout=100,
                            text=True,
                            check=True)
    rows = output.stdout.strip().split('\n')
    cols = fields.split(",")
    raw = pd.DataFrame([row.split("|")[:len(cols)]
                       for row in rows if row.count("|") > len(cols) - 2])
    raw.columns = cols
    return raw

In [4]:
def clean_dataframe(df: pd.DataFrame):
    """Clean the dataframe and set data types."""
    for col in ["elapsedraw", "start", "end"]:
        df = df[pd.notna(df[col])]
        df = df[df[col].str.isnumeric()]
        df[col] = df[col].astype("int64")
    df = df[df.elapsedraw > 0]
    return df

In [5]:
clusters = "della"
partition = ""
start_date = "2026-01-01T00:00:00"
end_date   = "now"
fields = "user,state,elapsedraw,alloctres,start,end"

# sacct for jdh4 only
df = get_data_from_sacct(clusters, partition, start_date, end_date, fields)
df = df[(df.state != "RUNNING") | (df.state != "PENDING")]
df = clean_dataframe(df)
df = df[(df.start > 1_000_000_000) & (df.end > 1_000_000_000)]

In [6]:
def billing_per_job(tres: str) -> int:
    """Return the billing."""
    bill = re.findall(r"billing=\d+", tres)
    return int(bill[0].replace("billing=", "")) if bill else 0

In [7]:
df["billing"] = df.alloctres.apply(billing_per_job)
df["billing-seconds"] = df["billing"] * df["elapsedraw"]

In [8]:
df

,user,state,elapsedraw,alloctres,start,end,billing,billing-seconds
0,ey7932,CANCELLED by 364704,2896,"billing=16,cpu=1,mem=64G,node=1",1770476331,1770479227,16,46336
1,ey7932,COMPLETED,89616,"billing=32,cpu=4,mem=128G,node=1",1770479297,1770568913,32,2867712
2,ey7932,COMPLETED,61266,"billing=32,cpu=4,mem=128G,node=1",1770752702,1770813968,32,1960512
3,ey7932,COMPLETED,59316,"billing=32,cpu=4,mem=128G,node=1",1770752702,1770812018,32,1898112
4,ey7932,COMPLETED,21,"billing=1,cpu=1,mem=4G,node=1",1772037695,1772037716,1,21
...,...,...,...,...,...,...,...,...
790,ey7932,COMPLETED,3433,"billing=12,cpu=2,mem=48G,node=1",1776119926,1776123359,12,41196
791,ey7932,CANCELLED by 364704,5525,"billing=8,cpu=1,mem=32G,node=1",1777491444,1777496969,8,44200
792,ey7932,CANCELLED by 364704,76,"billing=8,cpu=1,mem=32G,node=1",1777493183,1777493259,8,608
793,ey7932,FAILED,1,"billing=8,cpu=1,mem=32G,node=1",1777493548,1777493549,8,8


In [9]:
df["dt"] = round(time.time()) - df["end"]

In [10]:
round(df["billing-seconds"].sum())

25681980729

In [11]:
# scontrol show config | grep ^Priority
df["half-life"] = df["billing-seconds"] * 2**(-df["dt"] / (15 * 24 * 3600))

In [12]:
round(df["half-life"].sum())

3673438438

## Normalized Usage

In [14]:
# U = RawUsage (ey7932) / TotalUsage (root)
# use sshare -l to see NormUsage
U = 3646925410 / 1362680049452

## Normalized Shares

In [15]:
# S = RawShares (user) / RawShares (cluster)
S = 1 / 12029

## Fairshare

In [16]:
F = 2**(-U/S)